# Fase 3 — Deduplicación con MinHash + LSH

**Proyecto:** Clasificador Masivo de Noticias · **Equipo:** DataWhales
**Curso:** Datos Masivos I — Licenciatura en Ciencia de Datos, UNAM, 2026-2

---

## Objetivo

Detectar y eliminar noticias casi idénticas en un corpus de ~500k documentos usando:

1. **MinHash + LSH** para similitud **Jaccard** (sobre vectores binarios de tokens).
2. **Random Projection LSH** sobre vectores TF-IDF L2-normalizados como *proxy* para similitud **coseno**
   (para vectores normalizados, $d_E^2 = 2 - 2\cos\theta$, por lo que $d_E$ es monótona en $\cos\theta$).
3. **Union-Find** para agrupar duplicados en componentes conexas y conservar un solo representante por grupo.

## Entrada

`data/processed/` — Parquet con vectores TF-IDF de la Fase 2.
Esperado: `doc_id` (id único) y `features` (`SparseVector` TF-IDF). Se pueden ajustar nombres en la sección de configuración.

## Salida

- `data/dedup/corpus_deduplicado.parquet` — corpus limpio (un doc por cluster).
- `data/dedup/duplicate_pairs.parquet` — pares (`id_a`, `id_b`) detectados como duplicados.
- `data/dedup/duplicate_clusters.parquet` — mapa `doc_id → cluster_id`.

## 1. Setup de Spark e imports

In [1]:
import math
import time
import os
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, LongType, BooleanType
from pyspark.ml.feature import MinHashLSH, BucketedRandomProjectionLSH, Normalizer
from pyspark.ml.linalg import Vectors, SparseVector, VectorUDT

spark = (
    SparkSession.builder
    .appName("LSH_Dedup_Phase3")
    .master("local[2]")
    .config("spark.driver.memory", "6g")
    .config("spark.driver.maxResultSize", "2g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.parquet.compression.codec", "snappy")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark", spark.version)

26/06/06 19:35:13 WARN Utils: Your hostname, MacBook-Pro-de-Milena.local resolves to a loopback address: 127.0.0.1; using 192.168.1.66 instead (on interface en0)
26/06/06 19:35:13 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/06 19:35:14 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 3.5.1


## 2. Configuración

Ajusta rutas y umbrales aquí. Los valores por defecto son razonables para detección
de *near-duplicates* (no para clustering temático).

In [2]:
# El notebook vive en notebooks/ — subimos un nivel para llegar a la raiz
PROJECT_PATH = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATOS_PATH   = os.path.join(PROJECT_PATH, "datos")

# Rutas de entrada (salidas de Fase 2) 
INPUT_PATH = os.path.join(DATOS_PATH, "processed", "tokens_cc")

# Rutas de salida 
DEDUP_PATH    = os.path.join(DATOS_PATH, "dedup")
OUT_DEDUP     = os.path.join(DEDUP_PATH, "corpus_deduplicado.parquet")
OUT_PAIRS     = os.path.join(DEDUP_PATH, "duplicate_pairs.parquet")
OUT_CLUSTERS  = os.path.join(DEDUP_PATH, "duplicate_clusters.parquet")

os.makedirs(DEDUP_PATH, exist_ok=True)

# Columnas 
ID_COL       = "doc_id"
FEATURES_COL = "tf_vector"

# Hiperparametros MinHash / Jaccard 
NUM_HASH_TABLES_JACCARD = 3  
JACCARD_THRESHOLD       = 0.85 

# Hiperparametros Random Projection LSH / Coseno 
NUM_HASH_TABLES_COSINE = 3
COSINE_THRESHOLD       = 0.90
BUCKET_LENGTH          = 2.0

NUM_FEATURES = 65536
SEED = 42

print(f"INPUT_PATH → {INPUT_PATH}  ({'OK' if os.path.exists(INPUT_PATH) else 'FALTA'})")
print(f"OUT_DEDUP  → {OUT_DEDUP}")

INPUT_PATH → /Users/milenafer/Desktop/datos_masivos/Proyecto_Datos_Masivos/datos/processed/tokens_cc  (OK)
OUT_DEDUP  → /Users/milenafer/Desktop/datos_masivos/Proyecto_Datos_Masivos/datos/dedup/corpus_deduplicado.parquet


## 3. Carga del corpus TF-IDF de la Fase 2

In [3]:
from pyspark.ml.feature import HashingTF

df_raw = spark.read.parquet(INPUT_PATH)
n_total_raw = df_raw.count()
print(f"Documentos en disco: {n_total_raw:,}")

# Muestra de 10k docs — suficiente para demostrar MinHash LSH en local.
# approxSimilarityJoin es un self-join: 100k → horas, 10k → ~2-3 min.
df_sample = df_raw.filter(F.size(F.col("tokens")) > 0).limit(10_000)
n_total   = df_sample.count()
print(f"Muestra para LSH: {n_total:,} docs")

# binary_features para MinHash Jaccard (nativo JVM, sin Python UDF)
hashing_bin = HashingTF(inputCol="tokens", outputCol="binary_features",
                         numFeatures=NUM_FEATURES, binary=True)
# tf_vector para similitud coseno (Random Projection)
hashing_tf  = HashingTF(inputCol="tokens", outputCol=FEATURES_COL,
                         numFeatures=NUM_FEATURES, binary=False)

df = (hashing_tf.transform(hashing_bin.transform(df_sample))
      .select(ID_COL, FEATURES_COL, "binary_features")
      .cache())

n_cached = df.count()
print(f"Documentos vectorizados y cacheados: {n_cached:,}")

Documentos en disco: 100,000


Muestra para LSH: 10,000 docs


26/06/06 19:35:27 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


Documentos vectorizados y cacheados: 10,000


## 4. Preparar vectores binarios para MinHash

`MinHashLSH` interpreta los vectores como **conjuntos**: cualquier entrada ≠ 0 cuenta como
presencia de ese token. Como nuestros vectores TF-IDF ya son `SparseVector`, basta con
reemplazar los pesos por 1.0 y filtrar vectores totalmente vacíos (MinHash no los acepta).

In [4]:
# binary_features ya generados por HashingTF(binary=True) en cell-6 — nativo JVM, sin Python UDF
# df ya tiene binary_features y está cacheado
df_bin = df
n_bin  = n_cached

print(f"Documentos con al menos un token: {n_bin:,} (descartados: {n_total - n_bin:,})")

Documentos con al menos un token: 10,000 (descartados: 0)


## 5. MinHash + LSH — similitud Jaccard

`approxSimilarityJoin` devuelve pares cuya **distancia de Jaccard** ≤ umbral,
donde $d_J = 1 - J(A, B)$. Filtramos a $J \ge $ `JACCARD_THRESHOLD` y eliminamos
el self-join y los pares duplicados (a, b) / (b, a) con la condición `id_a < id_b`.

In [5]:
t0 = time.time()

mh = MinHashLSH(
    inputCol="binary_features",
    outputCol="hashes_jaccard",
    numHashTables=NUM_HASH_TABLES_JACCARD,
    seed=SEED)
mh_model = mh.fit(df_bin)
df_jac = mh_model.transform(df_bin).cache()
df_jac.count()

jaccard_pairs = (
    mh_model.approxSimilarityJoin(
        df_jac, df_jac,
        threshold=1.0 - JACCARD_THRESHOLD,
        distCol="jaccardDistance")
    .filter(F.col(f"datasetA.{ID_COL}") < F.col(f"datasetB.{ID_COL}"))
    .select(
        F.col(f"datasetA.{ID_COL}").alias("id_a"),
        F.col(f"datasetB.{ID_COL}").alias("id_b"),
        (F.lit(1.0) - F.col("jaccardDistance")).alias("jaccard_sim"))
).cache()

n_jac = jaccard_pairs.count()
print(f"[Jaccard ≥ {JACCARD_THRESHOLD}] pares candidatos: {n_jac:,}")
print(f"Tiempo Jaccard LSH: {time.time() - t0:.1f}s")
jaccard_pairs.orderBy(F.desc("jaccard_sim")).show(10, truncate=False)

[Jaccard ≥ 0.85] pares candidatos: 3,055
Tiempo Jaccard LSH: 136.5s
+-----------+-----------+-----------+
|id_a       |id_b       |jaccard_sim|
+-----------+-----------+-----------+
|8589944160 |8589950244 |1.0        |
|25769832152|25769832155|1.0        |
|12980      |25769888430|1.0        |
|34359762363|8590008038 |1.0        |
|8589940609 |8589944160 |1.0        |
|8589950244 |8589950705 |1.0        |
|20541      |8589968934 |1.0        |
|24139      |24155      |1.0        |
|8589935877 |8589950705 |1.0        |
|17179875242|34359809699|1.0        |
+-----------+-----------+-----------+
only showing top 10 rows



## 6. Random Projection LSH — proxy para similitud coseno

Spark MLlib no incluye LSH coseno nativo, pero para vectores **L2-normalizados** vale:

$$
\|x - y\|_2^2 = 2 - 2\cos\theta(x, y)
$$

por lo que ordenar por distancia euclidiana es equivalente a ordenar por coseno.
El umbral euclidiano correspondiente a $\cos\theta \ge \tau$ es $d_E \le \sqrt{2(1-\tau)}$.

In [6]:
t0 = time.time()

normalizer = Normalizer(inputCol=FEATURES_COL, outputCol="norm_features", p=2.0)
df_norm = normalizer.transform(df).cache()

brp = BucketedRandomProjectionLSH(
    inputCol="norm_features",
    outputCol="hashes_cosine",
    bucketLength=BUCKET_LENGTH,
    numHashTables=NUM_HASH_TABLES_COSINE,
    seed=SEED)
brp_model = brp.fit(df_norm)
df_cos = brp_model.transform(df_norm).cache()
df_cos.count()

eucl_threshold = math.sqrt(2.0 * (1.0 - COSINE_THRESHOLD))
print(f"Umbral euclidiano para cos ≥ {COSINE_THRESHOLD}: {eucl_threshold:.4f}")

cosine_pairs = (
    brp_model.approxSimilarityJoin(
        df_cos, df_cos,
        threshold=eucl_threshold,
        distCol="euclDistance")
    .filter(F.col(f"datasetA.{ID_COL}") < F.col(f"datasetB.{ID_COL}"))
    .select(
        F.col(f"datasetA.{ID_COL}").alias("id_a"),
        F.col(f"datasetB.{ID_COL}").alias("id_b"),
        (F.lit(1.0) - F.pow(F.col("euclDistance"), F.lit(2.0)) / F.lit(2.0)).alias("cosine_sim"))
).cache()

n_cos = cosine_pairs.count()
print(f"[Coseno ≥ {COSINE_THRESHOLD}] pares candidatos: {n_cos:,}")
print(f"Tiempo Cosine LSH: {time.time() - t0:.1f}s")
cosine_pairs.orderBy(F.desc("cosine_sim")).show(10, truncate=False)

26/06/06 19:37:54 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/06/06 19:37:54 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.VectorBLAS
26/06/06 19:37:55 WARN DAGScheduler: Broadcasting large task binary with size 1621.0 KiB
26/06/06 19:37:56 WARN DAGScheduler: Broadcasting large task binary with size 1626.2 KiB


Umbral euclidiano para cos ≥ 0.9: 0.4472


26/06/06 19:37:57 WARN DAGScheduler: Broadcasting large task binary with size 2020.8 KiB
26/06/06 19:46:52 WARN DAGScheduler: Broadcasting large task binary with size 2026.2 KiB


[Coseno ≥ 0.9] pares candidatos: 3,072
Tiempo Cosine LSH: 537.8s
+-----------+-----------+----------+
|id_a       |id_b       |cosine_sim|
+-----------+-----------+----------+
|17179917339|17179917758|1.0       |
|17179917339|17179917619|1.0       |
|17179917339|17179917393|1.0       |
|17179917339|17179917685|1.0       |
|17179910659|17179917339|1.0       |
|17179910659|17179912498|1.0       |
|17179910659|17179913005|1.0       |
|17179910659|17179913386|1.0       |
|17179910659|17179914366|1.0       |
|17179910659|17179915440|1.0       |
+-----------+-----------+----------+
only showing top 10 rows



26/06/06 19:46:52 WARN DAGScheduler: Broadcasting large task binary with size 2023.3 KiB


## 7. Unificar evidencia de Jaccard y Coseno

Un par se considera duplicado si **al menos una** de las dos medidas lo marca.
También guardamos las dos similitudes (cuando existen) en `duplicate_pairs.parquet`
para auditoría.

In [7]:
all_pairs = (
    jaccard_pairs.select("id_a", "id_b")
        .union(cosine_pairs.select("id_a", "id_b"))
        .distinct()
).cache()

# Pares enriquecidos con ambas metricas (left joins)
pairs_audit = (
    all_pairs
        .join(jaccard_pairs, ["id_a", "id_b"], "left")
        .join(cosine_pairs,  ["id_a", "id_b"], "left"))

n_all = all_pairs.count()
print(f"Pares duplicados únicos (Jaccard ∪ Coseno): {n_all:,}")
pairs_audit.show(10, truncate=False)

26/06/06 19:46:53 WARN DAGScheduler: Broadcasting large task binary with size 2.3 MiB
26/06/06 19:46:53 WARN DAGScheduler: Broadcasting large task binary with size 2.3 MiB
26/06/06 19:46:53 WARN DAGScheduler: Broadcasting large task binary with size 2.3 MiB


Pares duplicados únicos (Jaccard ∪ Coseno): 3,137
+-----------+-----------+------------------+------------------+
|id_a       |id_b       |jaccard_sim       |cosine_sim        |
+-----------+-----------+------------------+------------------+
|17179870095|755        |0.96              |0.9934731237349318|
|8589935877 |8589940609 |1.0               |1.0               |
|8589944160 |8589950244 |1.0               |1.0               |
|60129607293|60129607323|1.0               |0.9911639077268503|
|60129608652|60129608918|1.0               |1.0               |
|37399      |37426      |1.0               |1.0               |
|60129607797|60129608482|1.0               |0.9911639077268503|
|8589985295 |8589985298 |1.0               |1.0               |
|17179933505|77937      |0.8549019607843137|NULL              |
|60129607761|60129608918|1.0               |1.0               |
+-----------+-----------+------------------+------------------+
only showing top 10 rows



26/06/06 19:46:54 WARN DAGScheduler: Broadcasting large task binary with size 2025.3 KiB
26/06/06 19:46:54 WARN DAGScheduler: Broadcasting large task binary with size 2.3 MiB


## 8. Agrupar duplicados en clusters (Union-Find)

Un duplicado puede ser transitivo: si $A \sim B$ y $B \sim C$, los tres pertenecen
al mismo cluster. Construimos las **componentes conexas** del grafo de duplicados.

Como típicamente el número de pares post-LSH es pequeño comparado con $N$
(unos miles para 500k docs), traemos los pares al driver y usamos un Union-Find
en Python con compresión de caminos: $O(\alpha(n))$ amortizado por operación.

> Si el grafo crece demasiado para el driver, sustituye este bloque por
> `GraphFrames.connectedComponents()` o por *label propagation* iterativo en Spark.

In [8]:
class UnionFind:
    def __init__(self):
        self.parent = {}
        self.rank   = {}
    def find(self, x):
        if x not in self.parent:
            self.parent[x] = x
            self.rank[x]   = 0
            return x
        # path compression
        root = x
        while self.parent[root] != root:
            root = self.parent[root]
        while self.parent[x] != root:
            self.parent[x], x = root, self.parent[x]
        return root
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return
        if self.rank[ra] < self.rank[rb]:
            ra, rb = rb, ra
        self.parent[rb] = ra
        if self.rank[ra] == self.rank[rb]:
            self.rank[ra] += 1

pairs_local = all_pairs.collect()
print(f"Procesando {len(pairs_local):,} aristas en el driver…")

uf = UnionFind()
for row in pairs_local:
    uf.union(row.id_a, row.id_b)

cluster_map = {x: uf.find(x) for x in uf.parent}
n_dup_docs  = len(cluster_map)
n_clusters  = len(set(cluster_map.values()))
print(f"Documentos implicados en duplicados: {n_dup_docs:,}")
print(f"Clusters de duplicados: {n_clusters:,}")
print(f"Redundancia eliminable: {n_dup_docs - n_clusters:,} docs")

26/06/06 19:46:54 WARN DAGScheduler: Broadcasting large task binary with size 2.3 MiB


Procesando 3,137 aristas en el driver…
Documentos implicados en duplicados: 767
Clusters de duplicados: 241
Redundancia eliminable: 526 docs


## 9. Asignar cluster a cada documento y conservar un representante

- Los docs **no implicados** en ningún duplicado son singletons (`cluster_id = doc_id`).
- Para cada cluster conservamos un único representante. Aquí elegimos el de menor `doc_id`
  por determinismo; si tu pipeline tiene un criterio mejor (longitud, fecha de publicación,
  fuente más confiable), ajústalo en el `Window.orderBy(...)`.

In [9]:
bc_map = spark.sparkContext.broadcast(cluster_map)

# Tipo dinamico: respetamos el tipo del id original (string o numerico)
id_type = dict(df.dtypes)[ID_COL]

if id_type in ("bigint", "int", "long"):
    out_type = LongType()
    @F.udf(returnType=out_type)
    def get_cluster(doc_id):
        m = bc_map.value
        return int(m[doc_id]) if doc_id in m else int(doc_id)
else:
    out_type = StringType()
    @F.udf(returnType=out_type)
    def get_cluster(doc_id):
        m = bc_map.value
        return str(m[doc_id]) if doc_id in m else str(doc_id)

df_with_cluster = df.withColumn("cluster_id", get_cluster(F.col(ID_COL))).cache()

# Mapa explícito (útil para auditoría / Fase 5)
clusters_df = df_with_cluster.select(ID_COL, "cluster_id")

# Un representante por cluster (criterio: menor doc_id)
w = Window.partitionBy("cluster_id").orderBy(F.col(ID_COL).asc())
df_deduped = (
    df_with_cluster
        .withColumn("rn", F.row_number().over(w))
        .filter(F.col("rn") == 1)
        .drop("rn", "cluster_id")
)

n_deduped = df_deduped.count()
n_removed = n_total - n_deduped
print(f"Documentos originales       : {n_total:,}")
print(f"Documentos post-deduplicación: {n_deduped:,}")
print(f"Documentos eliminados       : {n_removed:,} ({100*n_removed/n_total:.2f}%)")

Documentos originales       : 10,000
Documentos post-deduplicación: 9,474
Documentos eliminados       : 526 (5.26%)


## 10. Escribir salidas a `data/dedup/`

In [10]:
(df_deduped
    .write.mode("overwrite")
    .parquet(OUT_DEDUP))

(pairs_audit
    .write.mode("overwrite")
    .parquet(OUT_PAIRS))

(clusters_df
    .write.mode("overwrite")
    .parquet(OUT_CLUSTERS))

print("Escrito:")
print(f"  • {OUT_DEDUP}")
print(f"  • {OUT_PAIRS}")
print(f"  • {OUT_CLUSTERS}")

26/06/06 19:47:00 WARN DAGScheduler: Broadcasting large task binary with size 2025.2 KiB
26/06/06 19:47:01 WARN DAGScheduler: Broadcasting large task binary with size 2.5 MiB


Escrito:
  • /Users/milenafer/Desktop/datos_masivos/Proyecto_Datos_Masivos/datos/dedup/corpus_deduplicado.parquet
  • /Users/milenafer/Desktop/datos_masivos/Proyecto_Datos_Masivos/datos/dedup/duplicate_pairs.parquet
  • /Users/milenafer/Desktop/datos_masivos/Proyecto_Datos_Masivos/datos/dedup/duplicate_clusters.parquet


## 11. Estadísticas y validación rápida

Distribución de tamaños de cluster (cuántos clusters de tamaño 2, 3, …) y un *spot-check*
de los pares con mayor similitud.

In [11]:
sizes = (
    clusters_df.groupBy("cluster_id").count()
    .groupBy("count").agg(F.count("*").alias("num_clusters"))
    .orderBy("count")
)
print("Distribución de tamaños de cluster:")
sizes.show(20, truncate=False)

print("Top pares con mayor similitud Jaccard:")
jaccard_pairs.orderBy(F.desc("jaccard_sim")).show(10, truncate=False)

print("Top pares con mayor similitud Coseno:")
cosine_pairs.orderBy(F.desc("cosine_sim")).show(10, truncate=False)

Distribución de tamaños de cluster:
+-----+------------+
|count|num_clusters|
+-----+------------+
|1    |9233        |
|2    |171         |
|3    |36          |
|4    |12          |
|5    |9           |
|6    |2           |
|8    |2           |
|9    |1           |
|11   |1           |
|12   |1           |
|14   |1           |
|16   |1           |
|25   |1           |
|30   |1           |
|38   |1           |
|41   |1           |
+-----+------------+

Top pares con mayor similitud Jaccard:
+-----------+-----------+-----------+
|id_a       |id_b       |jaccard_sim|
+-----------+-----------+-----------+
|8589944160 |8589950244 |1.0        |
|25769832152|25769832155|1.0        |
|12980      |25769888430|1.0        |
|34359762363|8590008038 |1.0        |
|8589940609 |8589944160 |1.0        |
|8589950244 |8589950705 |1.0        |
|20541      |8589968934 |1.0        |
|24139      |24155      |1.0        |
|8589935877 |8589950705 |1.0        |
|17179875242|34359809699|1.0        |
+---------

26/06/06 19:47:03 WARN DAGScheduler: Broadcasting large task binary with size 2023.3 KiB


## 12. Análisis de costo–comunicación (temario del curso)

Para $N$ documentos, $K$ funciones MinHash y $b$ bandas LSH (con $r = K/b$ filas por banda):

| Etapa | Costo computacional | Comunicación (shuffle) |
|---|---|---|
| MinHash (firma por doc) | $O(N \cdot K \cdot \overline{|D|})$ | $O(N \cdot K)$ |
| Banding y agrupación por bucket | $O(N \cdot b)$ | $O(N \cdot b)$ — *clave shuffle* |
| Verificación de pares candidatos | $O(C)$ con $C \ll N^2$ | $O(C)$ |
| Union-Find (driver) | $O(|E| \cdot \alpha(N))$ | $O(|E|)$ (collect) |

**Cota teórica del *trade-off* banding** (clase magistral): la probabilidad de que un par con
similitud Jaccard $s$ se vuelva candidato en al menos una banda es

$$
P(s) = 1 - (1 - s^{r})^{b}
$$

— curva sigmoide con punto de inflexión $\approx (1/b)^{1/r}$. Ajustar $(b, r)$ desplaza ese
umbral. En este notebook usamos `numHashTables = b` con $r = 1$ implícito por la API de Spark
(cada tabla es una banda independiente de un solo hash); aumentar `numHashTables` aumenta el
recall a costa de más candidatos a verificar.

**Comparativa vs. brute force:** la fuerza bruta exige $O(N^2)$ comparaciones — para
$N = 5\times 10^5$, son $\approx 1.25\times 10^{11}$ pares. LSH lo reduce a $O(N \cdot b)$ en la
fase de buckets más $O(C)$ verificaciones, donde $C$ es típicamente sublineal cuando los
duplicados son raros. **Es la mejora cualitativa que justifica usar LSH** en este pipeline.

## 13. Liberar caché

In [12]:
for d in [df, df_bin, df_jac, df_norm, df_cos, all_pairs, jaccard_pairs, cosine_pairs, df_with_cluster]:
    try:
        d.unpersist()
    except Exception:
        pass

print("Fase 3 completa. El corpus deduplicado queda listo para la Fase 4 (clasificador MLP).")

Fase 3 completa. El corpus deduplicado queda listo para la Fase 4 (clasificador MLP).
